Cluster-based permutation analysis for 2+ groups (based on F test).

Hypothesis: There are significant differences between body maps of **2+** different groups of participants.

Workflow
1. Load participant group assignments from Excel.
2. Load emotion maps from NPZ files.
3. Compute a pixel-wise omnibus F statistic.
4. Identify contiguous suprathreshold clusters.
5. Estimate cluster significance using permutation testing.
6. Save significant clusters and summary statistics.
7. Visualize group mean maps with significant clusters outlined.

Interpretation
A significant cluster indicates that at least one group differs
from another somewhere within that body region.

Because this is an omnibus ANOVA-style test, significant clusters
do not identify which specific groups differ. Follow-up
comparisons are required for pairwise interpretation.

In [103]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.stats import f as f_distribution
from datetime import datetime
from scipy.sparse import lil_matrix
from mne.stats import permutation_cluster_test
import pandas as pd
import subprocess

# ============================================================
# Statistical helper functions
# ============================================================

def approx_f_statistic(*args):
    """
    Compute an approximate pixel-wise omnibus F statistic.

    Parameters
    ----------
    *args
        One array per group with shape:
        (subjects, height, width)

    Returns
    -------
    ndarray
        F-like statistic map used by MNE's
        permutation_cluster_test.

    Notes
    -----
    Larger values indicate greater between-group differences.
    This statistic is used only for cluster formation and
    permutation inference.
    """
    groups = [np.asarray(g, dtype=float) for g in args]
    k = len(groups)
    n = np.array([g.shape[0] for g in groups])

    means = [g.mean(axis=0) for g in groups]
    vars_ = [g.var(axis=0, ddof=1) for g in groups]

    w = []
    for ni, v in zip(n, vars_):
        wi = np.zeros_like(v, dtype=float)
        valid_v = np.isfinite(v) & (v > 0)
        np.divide(ni, v, out=wi, where=valid_v)
        w.append(wi)

    sum_w = sum(w)
    valid = np.isfinite(sum_w) & (sum_w > 0)

    gm = np.zeros_like(sum_w, dtype=float)
    numer = sum(wi * mi for wi, mi in zip(w, means))
    np.divide(numer, sum_w, out=gm, where=valid)

    ss_between = sum(wi * (mi - gm) ** 2 for wi, mi in zip(w, means))
    ms_between = ss_between / (k - 1)

    safe_sum_w = np.where(valid, sum_w, 1.0)
    lam = sum((1 - wi / safe_sum_w) ** 2 / (ni - 1) for wi, ni in zip(w, n))
    denom = 1 + (2 * (k - 2) / (k**2 - 1)) * lam

    return np.where(valid, ms_between/denom, 0.0)

# ============================================================
# Connectivity matrix construction
# ============================================================

def create_8_connectivity(height, width):
    """
    Create 8-neighbor connectivity for a 2D image.

    Each pixel is connected to:
        up, down, left, right,
        and the four diagonal neighbors.
    """

    n_pixels = height * width

    adjacency = lil_matrix(
        (n_pixels, n_pixels),
        dtype=int,)

    for row in range(height):
        for col in range(width):

            pixel = row * width + col

            for row_offset in [-1, 0, 1]:
                for col_offset in [-1, 0, 1]:

                    # Don't connect a pixel to itself
                    if row_offset == 0 and col_offset == 0:
                        continue

                    neighbor_row = row + row_offset
                    neighbor_col = col + col_offset

                    # Make sure neighbor is inside image
                    if (
                        0 <= neighbor_row < height
                        and 0 <= neighbor_col < width):

                        neighbor = (neighbor_row * width + neighbor_col)

                        adjacency[pixel, neighbor] = 1

    return adjacency.tocsr()

# ============================================================
# Data loading functions
# ============================================================

def load_group_lookup(xlsx_path, id_column, group_column):
    """
    Load a subject_id -> group mapping from an Excel file.

    Parameters
    ----------
    xlsx_path : str or Path
        Path to the .xlsx file containing subject IDs and group labels.

    id_column : str
        Column name holding the subject/embody ID.

    group_column : str
        Column name holding the group label. Works for any number of
        distinct labels in this column (2, 3, 4, ...).

    Returns
    -------
    group_lookup : dict
        Maps subject_id (str) -> group (str)
    """
    xlsx_path = Path(xlsx_path)

    df = pd.read_excel(xlsx_path)

    if id_column not in df.columns:
        raise ValueError(
            f"Column '{id_column}' not found in {xlsx_path.name}. "
            f"Available columns: {list(df.columns)}")

    if group_column not in df.columns:
        raise ValueError(
            f"Column '{group_column}' not found in {xlsx_path.name}. "
            f"Available columns: {list(df.columns)}")

    # Convert IDs to strings so they match subject identifiers extracted from NPZ filenames.
    df[id_column] = df[id_column].astype(str).str.strip()
    df[group_column] = df[group_column].astype(str).str.strip()

    duplicate_ids = df[id_column][df[id_column].duplicated()]
    if not duplicate_ids.empty:
        print(f"Warning: duplicate subject IDs found: {duplicate_ids.tolist()}")

    missing_group = df[df[group_column].isin(["", "nan", "None"])]
    if not missing_group.empty:
        print(f"Warning: subjects with missing group assignment: {missing_group[id_column].tolist()}")

    group_lookup = dict(zip(df[id_column], df[group_column]))

    return group_lookup

def load_one_emotion_n_groups(npz_folder, emotion_index, group_lookup, group_list):
    """
    Load one emotion's maps for an arbitrary number of groups.

    Returns
    -------
    list of ndarray, one per entry in group_list, each shape
    (n_subjects_in_that_group, 600, 235).
    """
    npz_folder = Path(npz_folder)
    maps_by_group = {group: [] for group in group_list}

    for file_path in sorted(npz_folder.glob("*.npz")):
        subject_id = file_path.stem
        group = group_lookup.get(subject_id)
        if group not in group_list:
            continue

        with np.load(file_path) as saved:
            if "resmat" not in saved:
                continue
            # Extract only the current emotion map to reduce memory usage.
            # Each resmat contains all emotions, but only one is needed
            # for the current analysis.
            emotion_map = saved["resmat"][:, :, emotion_index].astype(np.float32)

        if emotion_map.shape != (600, 235):
            continue

        maps_by_group[group].append(emotion_map)

    for group in group_list:
        print(f"{group}: {len(maps_by_group[group])} subjects")

    return [np.stack(maps_by_group[group]) if maps_by_group[group] else np.empty((0, 600, 235))
            for group in group_list]

# ============================================================
# Cluster permutation analysis
# ============================================================

def cluster_permutation_test_n_groups(
    group_maps,
    group_names,
    n_permutations,
    adjacency,
    cluster_alpha=0.01,
    significance_alpha=0.05,
    n_jobs=-1,):
    """
    Omnibus cluster-based permutation test across 2+ groups (one-way ANOVA
    F-statistic in place of the two-sample t-statistic).
    """
    group_maps = [np.asarray(g, dtype=float) for g in group_maps]
    k = len(group_maps)

    n_total = sum(g.shape[0] for g in group_maps)

    dfn = k - 1
    dfd = n_total - k

    # Cluster-forming threshold.
    # Pixels exceeding this F value are eligible to become part
    # of a candidate cluster before permutation testing.
    f_threshold = f_distribution.ppf(1 - cluster_alpha, dfn, dfd)

    (observed_f_map,
        clusters,
        cluster_p_values,
        null_distribution,
    ) = permutation_cluster_test(
        group_maps,
        threshold=f_threshold,
        n_permutations=n_permutations,
        tail=1,
        stat_fun=approx_f_statistic,
        adjacency=adjacency,
        out_type="mask",
        n_jobs=n_jobs,
        verbose=True,)

    significant_mask = np.zeros(
        observed_f_map.shape,
        dtype=bool,)

    significant_clusters = []

    group_means = [np.nanmean(g, axis=0) for g in group_maps]

    for cluster_number, (
        cluster_mask,corrected_p,) in \
        (enumerate(zip(clusters, cluster_p_values),start=1,)):

        cluster_mask = np.asarray(
            cluster_mask,
            dtype=bool,)

        cluster_f_values = observed_f_map[
            cluster_mask]

        if cluster_f_values.size == 0:
            continue

        cluster_mass = np.sum(cluster_f_values)

        # Save average activation within the cluster for each group.
        # These values help interpret the direction and magnitude of
        # group differences after a significant omnibus result.
        per_group_cluster_means = {
            f"mean_{name}": float(np.nanmean(mean_map[cluster_mask]))
            for name, mean_map in zip(group_names, group_means)}

        cluster_result = {
            "number": cluster_number,
            "size": int(np.sum(cluster_mask)),
            "mass": float(cluster_mass),
            "p_value": float(corrected_p),
            "mask": cluster_mask,
            **per_group_cluster_means,}

        if corrected_p <= significance_alpha:
            significant_mask |= cluster_mask
            significant_clusters.append(cluster_result)

    return {
        "f_map": observed_f_map,
        "significant_mask": significant_mask,
        "significant_clusters": significant_clusters,}

# ============================================================
# Analysis across all emotions
# ============================================================

def analyze_all_emotions_n_groups(
    labels,
    n_permutations,
    adjacency,
    group_list,
    group_lookup,
    npz_folder,
    output_folder,
    results_path,
    cluster_alpha=0.01,
    significance_alpha=0.05,
    n_jobs=-1,):

    results_by_emotion = {}
    group_mean_maps = {}

    fieldnames = ([
        "emotion",
        "cluster_number",
        "size_pixels",
        "cluster_mass",
        "corrected_p_value",
        "significance_alpha",]
        + [f"mean_{group}" for group in group_list])

    for emotion_index, emotion_name in enumerate(labels):
        print(f"\n{'='*70}\nN-group test {emotion_index+1}/{len(labels)}: {emotion_name}\n{'='*70}")

        group_maps = load_one_emotion_n_groups(
            npz_folder, emotion_index, group_lookup, group_list)

        if any(g.shape[0] < 2 for g in group_maps):
            print(f"  Skipping {emotion_name}: at least one group has fewer than 2 subjects.")
            results_by_emotion[emotion_name] = {
                "f_map": np.full((600, 235), np.nan),
                "significant_mask": np.zeros((600, 235), dtype=bool),
                "significant_clusters": [],}

            group_mean_maps[emotion_name] = [
                np.nanmean(g, axis=0) if g.shape[0] > 0 else np.full((600, 235), np.nan)
                for g in group_maps]
            continue

        emotion_results = cluster_permutation_test_n_groups(
            group_maps=group_maps,
            group_names=group_list,
            n_permutations=n_permutations,
            adjacency=adjacency,
            cluster_alpha=cluster_alpha,
            significance_alpha=significance_alpha,
            n_jobs=n_jobs,)

        results_by_emotion[emotion_name] = emotion_results

        print(
            f"{emotion_name}: "
            f"{len(emotion_results['significant_clusters'])} "
            "significant clusters")

        for cluster in emotion_results["significant_clusters"]:
            print(
                f"  Cluster {cluster['number']}: "
                f"size={cluster['size']} pixels, "
                f"mass={cluster['mass']:.2f}, "
                f"corrected p={cluster['p_value']:.4f}")

        group_mean_maps[emotion_name] = [
            np.nanmean(g, axis=0) for g in group_maps]

        del group_maps

        cluster_rows = []

        for cluster in emotion_results["significant_clusters"]:
            row = {
                "emotion": emotion_name,
                "cluster_number": cluster["number"],
                "size_pixels": cluster["size"],
                "cluster_mass": cluster["mass"],
                "corrected_p_value": cluster["p_value"],
                "significance_alpha": significance_alpha,}

            for group in group_list:
                row[f"mean_{group}"] = cluster[f"mean_{group}"]
            cluster_rows.append(row)

        with open(results_path, "a", newline="", encoding="utf-8") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writerows(cluster_rows)

        output_path = output_folder / (f"cluster_analysis_{emotion_name}.npz")

        np.savez_compressed(
            output_path,
            f_map=emotion_results["f_map"],
            significant_mask=emotion_results["significant_mask"],
            group_mean_maps=np.array(group_mean_maps[emotion_name]),)

    return results_by_emotion, group_mean_maps

# ============================================================
# Visualization
# ============================================================

def visualize_n_group_cluster_results(
    group_mean_maps,results_by_emotion, base_image_path, labels, group_list,):
    """
    Display group mean maps for each emotion.

    Each panel shows the average body map for one group.
    Black dashed contours indicate clusters that survived
    cluster-based permutation testing.

    Because the underlying test is an omnibus F test,
    cluster outlines show where groups differ collectively,
    not which specific groups differ.
    """
    base_image = np.asarray(Image.open(base_image_path).convert("RGB"))
    base_crop = base_image[:, LEFT_X1:LEFT_X2, :]

    maximum = max(
        np.nanmax(np.abs(m))
        for emotion in labels
        for m in group_mean_maps[emotion])

    if not np.isfinite(maximum) or maximum == 0:
        maximum = 1

    n_groups = len(group_list)

    emotion_blocks_per_row = 2

    plot_rows = int(
        np.ceil(len(labels) / emotion_blocks_per_row))

    plot_columns = (emotion_blocks_per_row * n_groups)

    fig, axes = plt.subplots(
        plot_rows,
        plot_columns,
        figsize=(2* plot_columns, 4 * plot_rows),
        constrained_layout=False,)

    axes = np.atleast_2d(axes)

    heatmap = None

    row_tops = {}

    for row in range(plot_rows):
        row_tops[row] = max(
            axes[row, col].get_position().y1
            for col in range(plot_columns))

    for emotion_index, emotion_name in enumerate(labels):

        results = results_by_emotion[emotion_name]

        block_row = emotion_index // emotion_blocks_per_row
        block_col = emotion_index % emotion_blocks_per_row
        block_start_col = (block_col * n_groups)

        for group_index, group_name in enumerate(group_list):

            axis = axes[
                block_row,
                block_start_col + group_index]

            mean_map = (
                group_mean_maps
                [emotion_name]
                [group_index])

            axis.imshow(base_crop)

            heatmap = axis.imshow(
                mean_map,
                cmap="seismic",
                vmin=-maximum,
                vmax=maximum,
                alpha=0.65,
                interpolation="nearest",)

            if np.any(results["significant_mask"]):
                axis.contour(
                    results["significant_mask"]
                    .astype(float),
                    levels=[0.5],
                    colors=["black"],
                    linewidths=1.5,
                    linestyles="dashed")

            axis.set_title(group_name,fontsize=9,)

            axis.axis("off")

        left_ax = axes[block_row,block_start_col]
        right_ax = axes[block_row,block_start_col + n_groups - 1]

        bbox1 = left_ax.get_position()
        bbox2 = right_ax.get_position()

        x_center = (
            bbox1.x0 +
            bbox2.x1) / 2

        y_top = row_tops[block_row]

        fig.text(
            x_center,
            y_top + 0.01,
            f"{emotion_name}: {len(results['significant_clusters'])} clusters",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",)

    used_axes = len(labels) * n_groups

    for axis in axes.ravel()[used_axes:]:
        axis.axis("off")

    cbar_ax = fig.add_axes([0.90, 0.3, 0.015, 0.4])

    fig.colorbar(
        heatmap,
        cax=cbar_ax,
        label="Mean activation intensity" )

    fig.subplots_adjust(right=0.88,)

    plt.show()
    return fig

Change below

In [104]:
# Make sure it is same size used when creating resmat.
# Python excludes the ending index, so this crop is 235 pixels wide.
LEFT_X1 = 0
LEFT_X2 = 235

npz_folder = Path(
    "/Users/bb/Documents/Park lab/Embody/Preprocessed/test")

group_path = Path("/Users/bb/Documents/Park lab/Embody/Demographics/Complete_prolific_demographics.xlsx")

base_image_path = Path(
    "/Users/bb/Documents/Park lab/Embody/base.png")

output_folder = Path(
    "/Users/bb/Documents/Park lab/Embody/Results/test")

output_folder.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")

results_path = output_folder / (
    f"clusters_analysis_{run_timestamp}.csv")

LABELS = [
    'Neutral',
    'Fear',
    'Anger',
    'Disgust',
    'Sadness',
    'Happiness',
    'Lonely',
    'Anxiety',
    'Love',
    'Depression',
    'Pride',
    'Shame',
    'Jealousy',
    'Stress',
    'Migraine',
    'Nausea',
    ]

n_permutations = 1000

# Bonferroni correction across all tested emotions.
# Family-wise alpha = 0.05.
significance_alpha = 0.05/len(LABELS)

# Define the groups to compare.
#
# Values must match labels exactly as they appear in the
# demographics spreadsheet. The order specified here will
# determine display order in output tables and figures.
GROUP_LIST = ["18-28", "29-35", "36-46","47-80"]

Run below

In [105]:
# Start caffeinate in the background
caffeinate_proc = subprocess.Popen(['caffeinate', '-i'])

group_lookup = load_group_lookup(
    group_path,
    id_column="Embody_ID",
    group_column="age_group",)

adjacency = create_8_connectivity(600, 235)

with open(results_path, "w", newline="", encoding="utf-8") as csv_file:

    fieldnames = [
        "emotion",
        "cluster_number",
        "size_pixels",
        "cluster_mass",
        "corrected_p_value",
        "significance_alpha",
    ] + [f"mean_{group}" for group in GROUP_LIST]

    writer = csv.DictWriter(
        csv_file,
        fieldnames=fieldnames,)

    writer.writeheader()

results_by_emotion, group_mean_maps = analyze_all_emotions_n_groups(
    labels=LABELS,
    n_permutations=n_permutations,
    adjacency=adjacency,
    cluster_alpha=0.01,
    significance_alpha=significance_alpha,
    n_jobs=-1,
    group_list=GROUP_LIST,
    npz_folder=npz_folder,
    group_lookup=group_lookup,
    output_folder=output_folder,
    results_path=results_path,)

fig = visualize_n_group_cluster_results(
    group_mean_maps=group_mean_maps,
    results_by_emotion=results_by_emotion,
    base_image_path=base_image_path,
    labels=LABELS,
    group_list=GROUP_LIST,)

caffeinate_proc.terminate()


N-group test 1/16: Neutral
18-28: 84 subjects
29-35: 81 subjects
36-46: 72 subjects
47-80: 70 subjects
stat_fun(H1): min=0.0 max=4.300565528523108
Running initial clustering …
Found 1 cluster


100%|██████████| Permuting : 999/999 [00:33<00:00,   29.63it/s]


Neutral: 0 significant clusters

N-group test 2/16: Fear
18-28: 84 subjects
29-35: 81 subjects
36-46: 72 subjects
47-80: 70 subjects
stat_fun(H1): min=0.0 max=5.126452983214399
Running initial clustering …
Found 15 clusters


100%|██████████| Permuting : 999/999 [00:44<00:00,   22.30it/s]


Fear: 0 significant clusters

N-group test 3/16: Anger
18-28: 84 subjects
29-35: 81 subjects
36-46: 72 subjects
47-80: 70 subjects
stat_fun(H1): min=0.0 max=5.715829629777159
Running initial clustering …
Found 7 clusters


100%|██████████| Permuting : 999/999 [00:44<00:00,   22.68it/s]


Anger: 0 significant clusters

N-group test 4/16: Disgust
18-28: 84 subjects
29-35: 81 subjects
36-46: 72 subjects
47-80: 70 subjects
stat_fun(H1): min=0.0 max=3.885217080567398
Running initial clustering …
Found 1 cluster


 66%|██████▌   | Permuting : 658/999 [00:27<00:14,   24.06it/s]


KeyboardInterrupt: 